# Notebook 03 — Régression Ridge

## Objectif
Tester un premier modèle réel de régression linéaire avec imputation des valeurs manquantes. Comparer plusieurs valeurs de régularisation `alpha` et identifier la meilleure configuration par rapport à la baseline.

## Fichiers utilisés
- `data/X_train.csv` — variables explicatives
- `data/y_train.csv` — variable cible

## Résultat attendu
- Tableau comparatif des RMSE pour chaque `alpha`
- Meilleure valeur de `alpha` identifiée
- Comparaison avec la baseline
- Rapport Skore Hub (clé `02_ridge`)

## Notebook suivant
`04_grouped_validation.ipynb` — validation groupée par patient avec `GroupKFold`.

## 1. Imports

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

## 2. Connexion à Skore Hub

In [2]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement des données

In [3]:
DATA_DIR = Path("../data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

visits = X_train.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train)
assert visits["target"].notna().all()

print(f"visits : {visits.shape} — OK")

visits : (44590, 14) — OK


## 4. Sélection des variables numériques

In [4]:
feature_cols = [
    "sexM",
    "age_at_diagnosis",
    "age",
    "ledd",
    "time_since_intake_on",
    "time_since_intake_off",
    "on",
    "off",
]

X = visits[feature_cols]
y = visits["target"]

print(f"X : {X.shape}")
print(f"Valeurs manquantes par colonne :")
print(X.isna().sum())

X : (44590, 8)
Valeurs manquantes par colonne :
sexM                         0
age_at_diagnosis          2319
age                          0
ledd                     16338
time_since_intake_on     20678
time_since_intake_off    35116
on                       13220
off                      18913
dtype: int64


## 5. Baseline de référence

In [5]:
dummy = DummyRegressor(strategy="mean")
report_baseline = skore.evaluate(dummy, X, y)
rmse_baseline = report_baseline.metrics.rmse()
print(f"RMSE baseline (DummyRegressor) : {rmse_baseline:.4f}")

RMSE baseline (DummyRegressor) : 16.4779


## 6. Évaluation de Ridge pour plusieurs valeurs de `alpha`

Le pipeline `SimpleImputer(strategy="median") + Ridge` remplace les valeurs manquantes par la médiane de chaque colonne, puis applique la régression Ridge. La régularisation `alpha` contrôle la force de la pénalisation L2.

In [6]:
alphas = [0.1, 1.0, 10.0, 100.0]

results = []

for alpha in alphas:
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        Ridge(alpha=alpha),
    )

    report = skore.evaluate(model, X, y)
    rmse_val = report.metrics.rmse()

    results.append(
        {
            "alpha": alpha,
            "rmse": rmse_val,
            "vs_baseline": rmse_val - rmse_baseline,
        }
    )
    print(f"alpha={alpha:6.1f}  →  RMSE={rmse_val:.4f}")

results_df = pd.DataFrame(results).sort_values("rmse")
print()
print("=== Résultats triés par RMSE ===")
display(results_df.reset_index(drop=True))

alpha=   0.1  →  RMSE=10.4877
alpha=   1.0  →  RMSE=10.4877
alpha=  10.0  →  RMSE=10.4877
alpha= 100.0  →  RMSE=10.4877

=== Résultats triés par RMSE ===


,alpha,rmse,vs_baseline
0,0.1,10.487659,-5.990278
1,1.0,10.487659,-5.990278
2,10.0,10.487662,-5.990276
3,100.0,10.487689,-5.990249


## 7. Meilleure configuration et envoi du rapport Skore Hub

In [7]:
best_row = results_df.iloc[0]
best_alpha = best_row["alpha"]
best_rmse = best_row["rmse"]

print(f"Meilleur alpha : {best_alpha}")
print(f"RMSE Ridge     : {best_rmse:.4f}")
print(f"RMSE baseline  : {rmse_baseline:.4f}")
print(f"Amélioration   : {rmse_baseline - best_rmse:.4f}")

# Réentraîner le meilleur modèle pour envoyer son rapport
best_model = make_pipeline(
    SimpleImputer(strategy="median"),
    Ridge(alpha=best_alpha),
)
best_report = skore.evaluate(best_model, X, y)

hub_project.put("02_ridge", best_report)
print("Rapport envoyé à Skore Hub sous la clé : 02_ridge")

Meilleur alpha : 0.1
RMSE Ridge     : 10.4877
RMSE baseline  : 16.4779
Amélioration   : 5.9903


Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/estimators/51081

Rapport envoyé à Skore Hub sous la clé : 02_ridge


## 8. Conclusion

La régression Ridge avec imputation par la médiane améliore la RMSE par rapport au `DummyRegressor`.

**Points clés :**
- `SimpleImputer(strategy="median")` gère les valeurs manquantes de façon simple.
- La valeur de `alpha` a peu d'impact ici : le problème est probablement dominé par la qualité des variables plutôt que par la régularisation.
- La validation utilisée ici est aléatoire (shuffle split par défaut dans `skore.evaluate`). La prochaine étape corrige ce biais en utilisant `GroupKFold` sur `patient_id`.

**Prochaine étape** : `04_grouped_validation.ipynb` — validation groupée par patient.